In [1]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 634, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 634 (delta 152), reused 90 (delta 88), pack-reused 434 (from 3)
Receiving objects: 100% (634/634), 209.09 KiB | 19.01 MiB/s, done.
Resolving deltas: 100% (326/326), done.
Installing RAPIDS remaining 26.02 libraries
Using Python 3.12.13 environment at: /usr
Resolved 176 packages in 8.65s
Prepared 10 packages in 770ms
Uninstalled 4 packages in 214ms
Installed 10 packages in 45ms
 - bokeh==3.8.2
 + bokeh==3.6.3
 + cugraph-cu12==26.2.0
 + cuxfilter-cu12==26.2.0
 + datashader==0.19.1
 - holoviews==1.22.1
 + holoviews==1.20.2
 + jupyter-server-proxy==4.5.0
 - panel==1.9.0
 + panel==1.7.5
 + pyct==0.6.0
 - shapely==2.1.2
 + shapely==2.0.7
 + simpervisor==1.0.0

        ***********************************************************************
        The pip install of RAPIDS is complete.

        Please do n

In [9]:
import xgboost as xgb
import pandas as pd
import numpy as np
import cudf
import cuml
import seaborn as sns

from tensorflow import keras
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, balanced_accuracy_score
from cuml.ensemble import RandomForestClassifier as cuRF

In [3]:
df = pd.read_csv("dataset_completo_40036_v2.csv")

COLS_ELIMINAR = ['Tipo', 'L', 'B', 'A']
X = df.drop(COLS_ELIMINAR, axis=1)
y = df['Tipo']

class_counts = y.value_counts()
clases_validas = class_counts[class_counts >= 10].index
mask = y.isin(clases_validas)

X_filt = X[mask].copy()
y_filt = y[mask].copy()

clases_eliminadas = class_counts[class_counts < 10].index.tolist()

nan_cols = X_filt.columns[X_filt.isna().any()].tolist()


le = LabelEncoder()
y_enc = le.fit_transform(y_filt).astype(np.int32)


X_train, X_test, y_train, y_test = train_test_split(
    X_filt, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp).astype(np.float32)
X_test_scaled  = scaler.transform(X_test_imp).astype(np.float32)


In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(100,50,50),
    activation='tanh',
    solver='sgd',
    max_iter=1000,
    alpha = 0.001,
    random_state=42,
    learning_rate='adaptive',
)

from sklearn.utils.class_weight import compute_sample_weight
sw = compute_sample_weight('balanced', y_train)
mlp.fit(X_train_scaled, y_train)
predicciones = mlp.predict(X_test_scaled)

precision = accuracy_score(y_test, predicciones)
print(f"La precisión de la red es de: {precision * 100}%")

La precisión de la red es de: 99.92507492507492%


In [4]:
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method='hist',
    device='cuda',
    random_state=42
)

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv = 5, scoring='accuracy')

grid_search.fit(X_train_scaled, y_train)

print(f"Mejor configuración: {grid_search.best_params_}")
print(f"Mejor score de validación: {grid_search.best_score_:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:06:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [13:06:16] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:06:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain

Mejor configuración: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
Mejor score de validación: 0.9962


In [10]:
import tensorflow as tf
import numpy as np

# 1. Generar predicciones "suaves" del XGBoost sobre TODO el train
best_model = grid_search.best_estimator_
soft_labels = best_model.predict_proba(X_train_scaled)  # shape (n_samples, 7)

n_features = X_train_scaled.shape[1]
n_clases = soft_labels.shape[1]

# 2. Definir red Keras
model_keras = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(n_features,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(n_clases, activation='softmax')
])

model_keras.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Entrenar imitando al XGBoost
model_keras.fit(
    X_train_scaled, soft_labels,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# 4. Evaluar vs el XGBoost original
y_pred_keras = np.argmax(model_keras.predict(X_test_scaled), axis=1)
y_pred_xgb   = best_model.predict(X_test_scaled)

print(f"Accuracy Keras:   {accuracy_score(y_test, y_pred_keras):.4f}")
print(f"Accuracy XGBoost: {accuracy_score(y_test, y_pred_xgb):.4f}")

# 5. Guardar en formato Keras
model_keras.save('mejor_modelo.keras')

import json
with open('clases.json', 'w', encoding='utf-8') as f:
    json.dump(le.classes_.tolist(), f, ensure_ascii=False)
print(f"Clases guardadas ({len(le.classes_)}): {le.classes_.tolist()}")

Epoch 1/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.9475 - loss: 0.1708 - val_accuracy: 0.9788 - val_loss: 0.0553
Epoch 2/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9918 - loss: 0.0277 - val_accuracy: 0.9938 - val_loss: 0.0216
Epoch 3/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9950 - loss: 0.0205 - val_accuracy: 0.9950 - val_loss: 0.0172
Epoch 4/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9952 - loss: 0.0179 - val_accuracy: 0.9975 - val_loss: 0.0126
Epoch 5/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9967 - loss: 0.0145 - val_accuracy: 0.9984 - val_loss: 0.0107
Epoch 6/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9966 - loss: 0.0142 - val_accuracy: 0.9959 - val_loss: 0.0155
Epoch 7/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9974 - loss: 0.0139 - val_accuracy: 0.9991 - val_loss: 0.0092
Epoch 8/50
901/901 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9980 - loss: 0.0122 - val_accuracy: 0.

In [14]:
keras.models.load_model("mejor_modelo.keras")
sample = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9145, 0.3248, 0.0, 0.735, 0.7607, 1.0, 0.0, 0.0])
prediccion = model.predict(sample.reshape(1, -1))
print(prediccion)

XGBoostError: [14:13:07] /__w/xgboost/xgboost/src/learner.cc:806: Check failed: mparam_.num_feature != 0 (0 vs. 0) : 0 feature is supplied.  Are you using raw Booster interface?
Stack trace:
  [bt] (0) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x2c1a8c) [0x7cc7fd6c1a8c]
  [bt] (1) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x720c77) [0x7cc7fdb20c77]
  [bt] (2) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x726ce6) [0x7cc7fdb26ce6]
  [bt] (3) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(XGBoosterGetNumFeature+0x34) [0x7cc7fd5ca054]
  [bt] (4) /lib/x86_64-linux-gnu/libffi.so.8(+0x7e2e) [0x7cc87c294e2e]
  [bt] (5) /lib/x86_64-linux-gnu/libffi.so.8(+0x4493) [0x7cc87c291493]
  [bt] (6) /usr/lib/python3.12/lib-dynload/_ctypes.cpython-312-x86_64-linux-gnu.so(+0x98c1) [0x7cc87d5c88c1]
  [bt] (7) /usr/lib/python3.12/lib-dynload/_ctypes.cpython-312-x86_64-linux-gnu.so(+0x8ffe) [0x7cc87d5c7ffe]
  [bt] (8) 

In [ ]:
X_tr_gpu = cudf.DataFrame(X_train_scaled)
y_tr_gpu = cudf.Series(y_train)

rf = cuRF(
    n_estimators=500,
    max_depth=20,
    max_features='sqrt',
    bootstrap=True,
    n_streams=8,
    random_state=42
)
rf.fit(X_tr_gpu, y_tr_gpu)

X_te_gpu  = cudf.DataFrame(X_test_scaled)
preds_rf  = rf.predict(X_te_gpu).to_numpy()

print("=== RANDOM FOREST ===")
print(f"Accuracy balanceada: {balanced_accuracy_score(y_test, preds_rf):.4f}")
print(classification_report(y_test, preds_rf,
      target_names=[str(c) for c in le.classes_]))

xgb_model = xgb.XGBClassifier(
    tree_method='hist',
    device='cuda',
    eval_metric='mlogloss',
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    use_label_encoder=False,
    random_state=42
)
xgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=100
)

preds_xgb = xgb_model.predict(X_test_scaled)

print("\n=== XGBOOST ===")
print(f"Accuracy balanceada: {balanced_accuracy_score(y_test, preds_xgb):.4f}")
print(classification_report(y_test, preds_xgb,
      target_names=[str(c) for c in le.classes_]))

=== RANDOM FOREST ===
Accuracy balanceada: 0.9994
                   precision    recall  f1-score   support

    Miel Jaramago       1.00      1.00      1.00      1601
   Miel Sintética       1.00      1.00      1.00      1601
   Miel de Bosque       1.00      1.00      1.00      1601
   Miel de Retama       1.00      1.00      1.00      1602
Miel de milflores       1.00      1.00      1.00      1603

         accuracy                           1.00      8008
        macro avg       1.00      1.00      1.00      8008
     weighted avg       1.00      1.00      1.00      8008

[0]	validation_0-mlogloss:1.50149


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:09:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[100]	validation_0-mlogloss:0.03389
[200]	validation_0-mlogloss:0.00580
[300]	validation_0-mlogloss:0.00373
[400]	validation_0-mlogloss:0.00321
[499]	validation_0-mlogloss:0.00300

=== XGBOOST ===
Accuracy balanceada: 0.9991
                   precision    recall  f1-score   support

    Miel Jaramago       1.00      1.00      1.00      1601
   Miel Sintética       1.00      1.00      1.00      1601
   Miel de Bosque       1.00      1.00      1.00      1601
   Miel de Retama       1.00      1.00      1.00      1602
Miel de milflores       1.00      1.00      1.00      1603

         accuracy                           1.00      8008
        macro avg       1.00      1.00      1.00      8008
     weighted avg       1.00      1.00      1.00      8008



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ── 1. PCA para ver si las clases son separables ──────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Colores por clase
classes     = np.unique(y_train)
class_names = le.classes_
colors      = plt.cm.tab20(np.linspace(0, 1, len(classes)))

ax = axes[0]
for cls, color in zip(classes, colors):
    mask = y_train == cls
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=class_names[cls], alpha=0.3, s=5, color=color)
ax.set_title(f'PCA (varianza explicada: {pca.explained_variance_ratio_.sum():.1%})')
ax.legend(markerscale=3, bbox_to_anchor=(1.05, 1))

# ── 2. t-SNE para ver clusters reales ─────────────────────────────────
# Subsample para velocidad
idx    = np.random.choice(len(X_train_scaled), size=3000, replace=False)
X_sub  = X_train_scaled[idx]
y_sub  = y_train[idx]

tsne   = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_sub)

ax = axes[1]
for cls, color in zip(classes, colors):
    mask = y_sub == cls
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               label=class_names[cls], alpha=0.4, s=8, color=color)
ax.set_title('t-SNE (estructura real de los datos)')
ax.legend(markerscale=3, bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.savefig('separabilidad.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3. Correlación entre medias espectrales por clase ─────────────────
print("\n=== SIMILITUD ENTRE CLASES (correlación de espectros medios) ===")
medias = {}
for cls in classes:
    mask       = y_train == cls
    medias[class_names[cls]] = X_train_scaled[mask].mean(axis=0)

df_medias = pd.DataFrame(medias)
corr      = df_medias.corr()

import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0.8, vmax=1.0, ax=ax)
ax.set_title('Correlación entre espectros medios por clase\n(>0.99 = casi indistinguibles)')
plt.tight_layout()
plt.savefig('correlacion_clases.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'X_train_scaled' is not defined